# Exploración inicial de datos

## Habilitar entorno de ejecución

In [ ]:
# Ejecución de código para detectar si se está ejecutando en Google Colab
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
	print("Ejecutando en Google Colab")
else:
	print("Ejecurtando en un entorno local")

In [ ]:
# Importar bibliotecas necesarias
from pathlib import Path			# Para manejar rutas de archivos y directorios
import numpy as np					# Para procesar mascaras con etiquetas numéricas
import cv2							# Para procesar imágenes
import json							# Para procesar archivos JSON
import xml.etree.ElementTree as ET 	# Para procesar archivos XML
import matplotlib.pyplot as plt		# Para visualizar imágenes

In [ ]:
# Clonar el repositorio de GitHub si se está ejecutando en Google Colab
if IN_COLAB:
	# Indicar la liga del repositorio de GitHub a clonar
	REPO_URL = "https://github.com/bettypau-open-knowledge/del-dataset-a-unet_exploracion-inicial-datasets.git"
	# Indicar ruta que tendrá el proyecto clonado en Google Drive (revisar su nombre de acuerdo con el URL del repositorio)
	PROYECTO = Path("/content/del-dataset-a-unet_exploracion-inicial-datasets")
	if (PROYECTO / ".git").exists():
		print(f"Repositorio encontrado en: {PROYECTO}")
		print("Realizando un pull para actualizar el repositorio...")
		!git -C {PROYECTO} pull
	else:
		print(f"Clonando el repositorio:")
		!git clone https://github.com/bettypau-open-knowledge/del-dataset-a-unet_exploracion-inicial-datasets.git
else: # Si se está ejecutando en un entorno local, establecer la ruta del proyecto
	PROYECTO = Path("..")

print(f"Ruta del proyecto: {PROYECTO}")

## Exploración del árbol de archivos

In [ ]:
# Listar los archivos y directorios en la ruta del proyecto
list(PROYECTO.iterdir())

### ¿Y si quiero ver todo el árbol?

In [ ]:
def mostrar_arbol(ruta, max_nivel=None, prefijo="", nivel=0):
	# Detener la exploración al alcanzar el nivel máximo
	if max_nivel is not None and nivel >= max_nivel:
		return

	# Obtener y ordenar los elementos (carpetas primero)
	elementos = sorted(
		ruta.iterdir(),
		key=lambda x: (x.is_file(), x.name.lower()) # Minúsculas para que el sorted funcione en orden alfabetico.
	)

	for i, elemento in enumerate(elementos):
		ultimo = i == len(elementos) - 1

		conector = "└── " if ultimo else "├── "
		print(prefijo + conector + elemento.name)

		# Explorar subdirectorios recursivamente
		if elemento.is_dir() and not elemento.is_symlink(): # Evitar bucles infinitos al no explorar enlaces simbólicos.
			extension = "    " if ultimo else "│   "

			mostrar_arbol(
				elemento,
				max_nivel=max_nivel,
				prefijo=prefijo + extension,
				nivel=nivel + 1
			)

In [ ]:
mostrar_arbol(PROYECTO, max_nivel=1) # Mostrar el árbol de directorios hata un nivel de profundidad

In [ ]:
mostrar_arbol(PROYECTO) # Mostrar el árbol de directorios completo

## Exploración de archivos

Todos los archivos del dataset se encuentran en data/raw partiendo de la raiz del proyecto.

In [ ]:
DATA = PROYECTO / "data" / "raw"
print(f"Ruta de los datos: {DATA}")

### Ver los archivos por extensión

In [ ]:
list(DATA.rglob("*.jpg"))

In [ ]:
list(DATA.rglob("*.png"))

### Explorar archivos de metadatos

#### Archivo de datos general

In [ ]:
# Archivo de metadatos en raiz del directorio de datos
metadatos_general = DATA / "anotaciones.json"

In [ ]:
# Cargar el archivo
with metadatos_general.open("r", encoding="utf-8") as archivo:
	anotaciones = json.load(archivo)

# Verificar que se guardó un diccionario
print(type(anotaciones))

In [ ]:
def mostrar_arbol_json(datos, max_nivel=None, prefijo="", nivel=0):

	if max_nivel is not None and nivel >= max_nivel:
		return

	# Obtener elementos según el tipo de dato
	if isinstance(datos, dict):
		elementos = list(datos.items())

	elif isinstance(datos, list):
		elementos = list(enumerate(datos))

	else:
		return

	for i, (clave, valor) in enumerate(elementos):
		ultimo = i == len(elementos) - 1

		conector = "└── " if ultimo else "├── "

		# Mostrar claves y valores
		if isinstance(valor, (dict, list)):
			print(prefijo + conector + str(clave))
		else:
			print(prefijo + conector + f"{clave}: {valor}")

		# Explorar estructuras anidadas
		if isinstance(valor, (dict, list)):
			extension = "    " if ultimo else "│   "

			mostrar_arbol_json(
				valor,
				max_nivel=max_nivel,
				prefijo=prefijo + extension,
				nivel=nivel + 1
			)

In [ ]:
mostrar_arbol_json(anotaciones, max_nivel=1)

In [ ]:
mostrar_arbol_json(anotaciones)

#### Archivo de metadatos por imagen

In [ ]:
# Exploraremos el archivo de metadatos de Sphynx_19

nombre_clase = "Chihuahua"
nombre_imagen = "Chihuahua_190"

#nombre_clase = "Maine_Coon"
#nombre_imagen = "Maine_Coon_186"

#nombre_clase = "Siamese"
#nombre_imagen = "Siamese_101"

#nombre_clase = "Sphynx"
#nombre_imagen = "Sphynx_19"

In [ ]:
# Construir la ruta del archivo de metadatos de la imagen
metadatos_imagen = DATA / "images" / nombre_clase / f"{nombre_imagen}.xml"
print(f"Ruta del archivo de metadatos de {nombre_imagen}: {metadatos_imagen}")

In [ ]:
# Cargar el archivo
arbol_xml = ET.parse(metadatos_imagen)

# Obtener el elemento raíz
raiz = arbol_xml.getroot()

# Verificar que se guardó un elemento raíz
print(type(raiz))
print(f"Elemento raíz: {raiz.tag}")

In [ ]:
def mostrar_arbol_xml(elemento, max_nivel=None, prefijo="", nivel=0):

	# Detener la exploración al alcanzar el nivel máximo
	if max_nivel is not None and nivel >= max_nivel:
		return

	# Obtener los elementos hijos
	elementos = list(elemento)

	for i, hijo in enumerate(elementos):
		ultimo = i == len(elementos) - 1

		conector = "└── " if ultimo else "├── "

		# Obtener el contenido de texto del elemento
		valor = (hijo.text or "").strip()

		# Mostrar etiqueta y valor, cuando corresponda
		if len(hijo) == 0 and valor:
			print(prefijo + conector + f"{hijo.tag}: {valor}")
		else:
			print(prefijo + conector + hijo.tag)

		# Explorar elementos hijos recursivamente
		if len(hijo) > 0:
			extension = "    " if ultimo else "│   "

			mostrar_arbol_xml(
				hijo,
				max_nivel=max_nivel,
				prefijo=prefijo + extension,
				nivel=nivel + 1
			)

In [ ]:
# Mostrar el árbol XML hasta un nivel de profundidad
print(raiz.tag)
mostrar_arbol_xml(raiz, max_nivel=1)

In [ ]:
# Mostrar el árbol XML completo
print(raiz.tag)
mostrar_arbol_xml(raiz)

NOTA!
Al trabajar con XML, se guardan los valores como cadenas, por lo que es necesario cambiar el tipo de dato a los campos numéricos (e.j. int()).

### Exploración inicial de imágenes

In [ ]:
# Elegimos la misma imagen de la sección anterior
# para explorar la imagen fotográfica y su máscara de segmentación.
ruta_imagen = DATA / "images" / nombre_clase / f"{nombre_imagen}.jpg"
ruta_mascara = DATA / "images" / nombre_clase / f"{nombre_imagen}.png"
print(f"Ruta de la imagen: {ruta_imagen}")
print(f"Ruta de la máscara: {ruta_mascara}")

In [ ]:
# Verificar que el archivo exista
if ruta_imagen.is_file():

	# Cargar la imagen con OpenCV (formato BGR)
	imagen_bgr = cv2.imread(str(ruta_imagen))

	if imagen_bgr is not None: # Si se logró abrir la imagen

		# Convertir de BGR a RGB
		imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)

		# Mostrar información de la imagen
		print("Dimensiones:", imagen_rgb.shape)
		print("Tipo de dato:", imagen_rgb.dtype)
		print("Valores únicos:", np.unique(imagen_rgb))
		
		# Visualizar
		plt.figure(figsize=(7, 5))
		plt.imshow(imagen_rgb)
		plt.title(ruta_imagen.name)
		plt.axis("off")
		plt.show()

	else: # Si no se logró abrir la imagen
		print("OpenCV no pudo leer la imagen.")

else: # Si el archivo no existe
	print(f"No se encontró la imagen: {ruta_imagen.resolve()}")

In [ ]:
# Verificar que el archivo exista
if ruta_mascara.is_file():

	# Cargar la máscara conservando sus valores originales
	mascara = cv2.imread(str(ruta_mascara), cv2.IMREAD_UNCHANGED)

	if mascara is not None: # Si se logró abrir la imagen

		# Mostrar información de la máscara
		print("Dimensiones:", mascara.shape)
		print("Tipo de dato:", mascara.dtype)
		print("Valores únicos:", np.unique(mascara))

		# Visualizar
		plt.figure(figsize=(7, 5))
		plt.imshow(mascara, cmap="gray", vmin=1, vmax=3)
		plt.title(ruta_mascara.name)
		plt.axis("off")
		plt.show()

	else: # Si no se logró abrir la imagen
		print("OpenCV no pudo leer la máscara.")

else: # Si el archivo no existe
	print(f"No se encontró la máscara: {ruta_mascara.resolve()}")

In [ ]:
# Verificar que imagen y máscara tengan las mismas dimensiones
if imagen_rgb.shape[:2] == mascara.shape[:2]:

	# Crear una imagen RGB del mismo tamaño que la original
	mascara_color = np.zeros_like(imagen_rgb)

   # Asignar colores a las etiquetas
	mascara_color[mascara == 1] = (0, 229, 255)    # Animal: cian
	mascara_color[mascara == 2] = (163, 163, 163)  # Fondo: gris
	mascara_color[mascara == 3] = (255, 0, 204)    # Contorno: magenta

	# Definir transparencia
	alpha = 0.4

	# Combinar imagen original y máscara coloreada
	superposicion = cv2.addWeighted(
		imagen_rgb, 1 - alpha,
		mascara_color, alpha,
		0
	)

	# Visualizar
	plt.figure(figsize=(7, 5))
	plt.imshow(superposicion)
	plt.title("Imagen con máscara superpuesta")
	plt.axis("off")
	plt.show()

else:
	print("La imagen y la máscara tienen dimensiones diferentes.")

In [ ]:
# Agregar bounding box de la cabeza de la estructura de la sección anterior
# Extraer las coordenadas del bounding box
xmin = int(raiz.findtext("object/bndbox/xmin"))
ymin = int(raiz.findtext("object/bndbox/ymin"))
xmax = int(raiz.findtext("object/bndbox/xmax"))
ymax = int(raiz.findtext("object/bndbox/ymax"))

# Obtener la clase del objeto
clase = raiz.findtext("object/name")

# Mostrar información
print("Clase:", clase)
print("Bounding box:", xmin, ymin, xmax, ymax)

In [ ]:
# Crear una copia de la imagen original
imagen_bbox = imagen_rgb.copy()

# Dibujar el bounding box
cv2.rectangle(
    imagen_bbox,
    (xmin, ymin),        # Esquina superior izquierda
    (xmax, ymax),        # Esquina inferior derecha
    color=(255, 0, 204), # Magenta (RGB)
    thickness=3          # Grosor del borde
)

# Visualizar
plt.figure(figsize=(7, 5))
plt.imshow(imagen_bbox)
plt.title(f"Bounding box de la cabeza: {clase}")
plt.axis("off")
plt.show()